# Lightning Prediction with Dual-CNN Architecture

This notebook implements a dual-CNN approach for predicting variable-length sequences of lightning strikes from multimodal storm imagery.

**Architecture:**
- **Shared 3D Encoder**: 3D CNN to extract spatial-temporal features from all 4 modalities
- **Spatial Head**: MLP to predict (x, y) coordinates for each strike slot
- **Temporal Head**: 1D CNN across strike slots to predict time (t) values
- **Stop Head**: Predicts when to stop outputting strikes for variable-length output

In [1]:
# IMPORTANT:
#   Whenever you start a new colab runtime, use the following code to download
#   the training dataset onto the runtime local storage.
#   This should take ~3-5 mins for the whole dataset.
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="benmoseley/ese-dl-2025-26-group-project", filename="train.h5", repo_type="dataset", local_dir="data")
hf_hub_download(repo_id="benmoseley/ese-dl-2025-26-group-project", filename="events.csv", repo_type="dataset", local_dir="data")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


train.h5:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

events.csv: 0.00B [00:00, ?B/s]

'data/events.csv'

In [2]:
import pandas as pd
import h5py
import IPython
import PIL
from PIL import Image
import matplotlib.pyplot as plt
import os
import io
import numpy as np
import random
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from IPython.display import clear_output

In [3]:
# Load events metadata
df = pd.read_csv("data/events.csv")
df

,id,img_type,event_type,start_utc,llcrnrlat,llcrnrlon,urcrnrlat,urcrnrlon,proj,height_m,width_m
0,S778114,ir069,Hail,2018-08-20 19:50:00+00:00,32.781533,-92.631841,35.944590,-88.134075,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
1,S778114,ir107,Hail,2018-08-20 19:50:00+00:00,32.781533,-92.631841,35.944590,-88.134075,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
2,S778114,lght,Hail,NaN,32.781533,-92.631841,35.944590,-88.134075,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
3,S778114,vil,Hail,2018-08-20 19:50:00+00:00,32.781533,-92.631841,35.944590,-88.134075,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
4,S778114,vis,Hail,2018-08-20 19:50:00+00:00,32.781533,-92.631841,35.944590,-88.134075,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
...,...,...,...,...,...,...,...,...,...,...,...
3995,S832818,ir069,Funnel Cloud,2019-07-28 17:44:00+00:00,46.493514,-100.371384,49.953392,-95.191216,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
3996,S832818,ir107,Funnel Cloud,2019-07-28 17:44:00+00:00,46.493514,-100.371384,49.953392,-95.191216,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
3997,S832818,lght,Funnel Cloud,NaN,46.493514,-100.371384,49.953392,-95.191216,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0
3998,S832818,vil,Funnel Cloud,2019-07-28 17:45:00+00:00,46.493514,-100.371384,49.953392,-95.191216,+proj=laea +lat_0=38 +lon_0=-98 +units=m +a=63...,384000.0,384000.0


In [4]:
# This function loads all of the image arrays for a given id
def load_event(id):
    "Load event"
    with h5py.File(f'data/train.h5','r') as f:
        event = {img_type: f[id][img_type][:] for img_type in ['vis', 'ir069', 'ir107', 'vil', 'lght']}
    return event

# Test loading
event = load_event("S778114")
for img_type in event:
    print(f"{img_type}: {event[img_type].shape} ({event[img_type].dtype})")
print(f"Event type: {df[df.id == 'S778114'].iloc[0].event_type}")

vis: (384, 384, 36) (int16)
ir069: (192, 192, 36) (int16)
ir107: (192, 192, 36) (int16)
vil: (384, 384, 36) (uint8)
lght: (38777, 5) (float32)
Event type: Hail


## Dataset and Data Loading

Uses PyTorch `Dataset` class for proper data loading:
- `StormDataset`: Loads and preprocesses events on-the-fly from HDF5 file
- Resize vis and vil (384×384) to match ir069 and ir107 (192×192)
- Normalize each modality to approximately [0, 1]
- Normalize targets (t, x, y) to [0, 1] for balanced training
- Custom collate function handles variable-length targets with padding and masking

In [5]:
class StormDataset(Dataset):
    """
    PyTorch Dataset for storm prediction.
    Loads and preprocesses storm events on-the-fly.
    
    Args:
        event_ids: List of event IDs to include in this dataset
        data_dir: Directory containing train.h5 and events.csv
    """
    
    def __init__(self, event_ids, data_dir="data"):
        self.event_ids = event_ids
        self.data_dir = data_dir
        self.h5_path = f"{data_dir}/train.h5"
    
    def __len__(self):
        return len(self.event_ids)
    
    def __getitem__(self, idx):
        event_id = self.event_ids[idx]
        
        # Load event data
        with h5py.File(self.h5_path, 'r') as f:
            event = {img_type: f[event_id][img_type][:] for img_type in ['vis', 'ir069', 'ir107', 'vil', 'lght']}
        
        # Process images
        images = self._process_images(event)
        
        # Process targets
        targets = self._process_targets(event)
        
        return images, targets, event_id
    
    def _process_images(self, event):
        """Resize and normalize image modalities."""
        # Initialize arrays for resized images (192x192)
        vis_resized = np.zeros((192, 192, 36), dtype=event['vis'].dtype)
        vil_resized = np.zeros((192, 192, 36), dtype=event['vil'].dtype)
        
        for i in range(36):
            vis_resized[:, :, i] = np.array(
                Image.fromarray(event['vis'][:, :, i]).resize((192, 192), Image.BILINEAR)
            )
            vil_resized[:, :, i] = np.array(
                Image.fromarray(event['vil'][:, :, i]).resize((192, 192), Image.BILINEAR)
            )
        
        # ir069 and ir107 are already 192x192
        ir069_data = event['ir069']
        ir107_data = event['ir107']
        
        # Permute to (frames, height, width)
        vis_permuted = np.transpose(vis_resized, (2, 0, 1))
        ir069_permuted = np.transpose(ir069_data, (2, 0, 1))
        ir107_permuted = np.transpose(ir107_data, (2, 0, 1))
        vil_permuted = np.transpose(vil_resized, (2, 0, 1))
        
        # Normalize each channel to ~[0, 1]
        vis_norm = vis_permuted / 10000.0
        ir069_norm = (ir069_permuted + 8000.0) / 7000.0
        ir107_norm = (ir107_permuted + 7000.0) / 9000.0
        vil_norm = vil_permuted / 255.0
        
        # Stack: (channels=4, frames=36, height=192, width=192)
        images = np.stack([vis_norm, ir069_norm, ir107_norm, vil_norm], axis=0).astype(np.float32)
        
        return images
    
    def _process_targets(self, event):
        """Normalize lightning targets (t, x, y) to [0, 1]."""
        # Scale coordinates from 384 to 192
        original_x = event['lght'][:, 3]
        original_y = event['lght'][:, 4]
        resized_x = original_x / 2.0
        resized_y = original_y / 2.0
        
        # Normalize to [0, 1]
        t_norm = event['lght'][:, 0] / 10800.0  # 0-10800 seconds
        x_norm = resized_x / 192.0
        y_norm = resized_y / 192.0
        
        targets = np.stack([t_norm, x_norm, y_norm], axis=1).astype(np.float32)
        
        return targets


def create_datasets(data_dir="data", test_size=0.2, val_size=0.1, subset_size=None, random_seed=42):
    """
    Create train/validation/test StormDataset instances.
    
    Args:
        data_dir: Directory containing train.h5 and events.csv
        test_size: Fraction of data for test set
        val_size: Fraction of data for validation set
        subset_size: If provided, only use this many events (for debugging)
        random_seed: Random seed for reproducibility
    
    Returns:
        train_dataset, val_dataset, test_dataset: StormDataset instances
    """
    df = pd.read_csv(f"{data_dir}/events.csv")
    all_ids = df['id'].unique().tolist()
    
    if subset_size:
        random.seed(random_seed)
        all_ids = random.sample(all_ids, subset_size)
    
    # Split data
    train_val_ids, test_ids = train_test_split(all_ids, test_size=test_size, random_state=random_seed)
    val_size_adjusted = val_size / (1 - test_size)
    train_ids, val_ids = train_test_split(train_val_ids, test_size=val_size_adjusted, random_state=random_seed)
    
    # Create datasets
    train_dataset = StormDataset(train_ids, data_dir)
    val_dataset = StormDataset(val_ids, data_dir)
    test_dataset = StormDataset(test_ids, data_dir)
    
    return train_dataset, val_dataset, test_dataset

In [ ]:
def collate_fn(batch, max_strikes=50000):
    """
    Collate function that creates proper padding and masks for variable-length sequences.
    Handles variable-length lightning sequences by padding to the maximum length in the batch.
    
    Args:
        batch: List of (images, targets, event_id) tuples from StormDataset
        max_strikes: Maximum number of strikes to keep per sample
    
    Returns:
        images: (B, 4, 36, 192, 192) - stacked image tensors
        targets_padded: (B, max_len, 3) - padded target sequences
        target_mask: (B, max_len) - True for real strikes, False for padding
        strike_counts: (B,) - actual number of strikes per sample
        ids: tuple of event IDs
    """
    images, targets, ids = zip(*batch)
    
    # Get actual counts and cap at max_strikes
    strike_counts = [min(t.shape[0], max_strikes) for t in targets]
    max_len = max(strike_counts) if strike_counts else 1
    max_len = max(max_len, 1)  # At least 1
    
    B = len(targets)
    targets_padded = torch.zeros(B, max_len, 3)
    target_mask = torch.zeros(B, max_len, dtype=torch.bool)
    
    for i, (t, count) in enumerate(zip(targets, strike_counts)):
        if count > 0:
            targets_padded[i, :count] = torch.tensor(t[:count])
            target_mask[i, :count] = True
    
    images = torch.stack([torch.tensor(img) for img in images])
    strike_counts = torch.tensor(strike_counts, dtype=torch.int32)
    
    return images, targets_padded, target_mask, strike_counts, ids


def create_dataloaders(train_dataset, val_dataset, test_dataset, batch_size=4, max_strikes=50000, num_workers=0):
    """
    Create DataLoaders from StormDataset instances.
    
    Args:
        train_dataset: StormDataset for training
        val_dataset: StormDataset for validation
        test_dataset: StormDataset for testing
        batch_size: Batch size for all loaders
        max_strikes: Maximum strikes per sample (passed to collate function)
        num_workers: Number of worker processes for data loading
    
    Returns:
        train_loader, val_loader, test_loader: DataLoader instances
    """
    collate = lambda b: collate_fn(b, max_strikes)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate,
        num_workers=num_workers
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=collate,
        num_workers=num_workers
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=collate,
        num_workers=num_workers
    )
    
    return train_loader, val_loader, test_loader

## Training

In [ ]:
# Create datasets (adjust subset_size based on your compute resources)
train_dataset, val_dataset, test_dataset = create_datasets(subset_size=200)
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset, val_dataset, test_dataset, 
    batch_size=4, 
    max_strikes=50000  # Default: 50000, handles all events in dataset
)

print(f"Training events: {len(train_dataset)}")
print(f"Validation events: {len(val_dataset)}")
print(f"Test events: {len(test_dataset)}")

# Check batch shapes
for images, targets, mask, strike_counts, ids in train_loader:
    print(f"\nBatch shapes:")
    print(f"  Images: {images.shape}")
    print(f"  Targets: {targets.shape}")
    print(f"  Mask: {mask.shape}")
    print(f"  Strike counts: {strike_counts}")
    break

## Model Architecture

Dual-CNN architecture for predicting variable-length sequences of lightning strikes:

```
Images → SharedEncoder(3D CNN) → Global Feature Vector
                                       │
        ┌──────────────────────────────┴──────────────────────────────┐
        │                                                              │
   SpatialHead                                                   TemporalHead
   (MLP → (x,y))                                                (1D Conv → t)
        │                                                              │
   (B, N, 2)                                                      (B, N, 1)
        └──────────────────────────────┬──────────────────────────────┘
                                       │
                              Combined: (B, N, 3) = (t, x, y)
                                       +
                              Stop logits: (B, N) → when to stop
```

**Key features:**
- **Parallel prediction**: All strike slots predicted at once
- **3D CNN encoder**: Extracts spatial-temporal features from all 4 modalities
- **Separate heads**: Spatial (MLP) and temporal (1D Conv) prediction heads
- **Variable-length output**: Stop token mechanism for variable-length sequences

In [ ]:
class SharedEncoder(nn.Module):
    """
    3D CNN encoder shared between spatial and temporal prediction heads.
    
    Input: (B, 4, 36, 192, 192)
    Output: (B, feature_dim) global feature vector
    """
    def __init__(self, in_channels=4, feature_dim=512):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            # (B, 4, 36, 192, 192) → (B, 32, 36, 48, 48)
            nn.Conv3d(in_channels, 32, kernel_size=(3, 5, 5), stride=(1, 4, 4), padding=(1, 2, 2)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            
            # (B, 32, 36, 48, 48) → (B, 64, 36, 12, 12)
            nn.Conv3d(32, 64, kernel_size=(3, 5, 5), stride=(1, 4, 4), padding=(1, 2, 2)),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            
            # (B, 64, 36, 12, 12) → (B, 128, 18, 6, 6)
            nn.Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1)),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            
            # (B, 128, 18, 6, 6) → (B, 256, 9, 3, 3)
            nn.Conv3d(128, 256, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1)),
            nn.BatchNorm3d(256),
            nn.ReLU(),
        )
        
        # Global pooling to fixed-size feature vector
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc = nn.Linear(256, feature_dim)
    
    def forward(self, x):
        features = self.conv_layers(x)           # (B, 256, 9, 3, 3)
        pooled = self.global_pool(features)      # (B, 256, 1, 1, 1)
        pooled = pooled.view(pooled.size(0), -1) # (B, 256)
        return self.fc(pooled)                   # (B, feature_dim)


class SpatialHead(nn.Module):
    """
    Predicts (x, y) normalized coordinates for N strike slots.
    
    Input: (B, feature_dim)
    Output: (B, max_strikes, 2) - (x, y) in [0, 1]
    """
    def __init__(self, feature_dim=512, max_strikes=50000, hidden_dim=256):
        super().__init__()
        self.max_strikes = max_strikes
        
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, max_strikes * 2),  # x, y per slot
        )
    
    def forward(self, features):
        out = self.mlp(features)                    # (B, max_strikes * 2)
        out = out.view(-1, self.max_strikes, 2)     # (B, max_strikes, 2)
        return torch.sigmoid(out)                    # Normalize to [0, 1]


class TemporalHead(nn.Module):
    """
    Predicts time (t) for N strike slots.
    Uses 1D convolution across strike slots to learn temporal ordering.
    
    Input: (B, feature_dim)
    Output: (B, max_strikes, 1) - t in [0, 1]
    """
    def __init__(self, feature_dim=512, max_strikes=50000):
        super().__init__()
        self.max_strikes = max_strikes
        
        # Project to per-slot features
        self.fc1 = nn.Linear(feature_dim, max_strikes * 16)
        
        # 1D conv across strike slots to learn temporal patterns
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Conv1d(16, 1, kernel_size=3, padding=1),
        )
    
    def forward(self, features):
        B = features.size(0)
        
        # Project to per-slot features
        out = self.fc1(features)                           # (B, max_strikes * 16)
        out = out.view(B, self.max_strikes, 16)            # (B, max_strikes, 16)
        out = out.permute(0, 2, 1)                         # (B, 16, max_strikes)
        
        # Temporal conv across slots
        out = self.temporal_conv(out)                      # (B, 1, max_strikes)
        out = out.permute(0, 2, 1)                         # (B, max_strikes, 1)
        
        return torch.sigmoid(out)                          # Normalize to [0, 1]


class DualCNNLightningPredictor(nn.Module):
    """
    Dual-CNN model that predicts variable-length (t, x, y) coordinates.
    
    Architecture:
    - Shared 3D encoder extracts features from input
    - Spatial head predicts (x, y) for each strike slot
    - Temporal head predicts (t) for each strike slot
    - Stop head predicts when to stop outputting strikes
    """
    def __init__(self, in_channels=4, feature_dim=512, max_strikes=50000):
        super().__init__()
        self.max_strikes = max_strikes
        
        self.encoder = SharedEncoder(in_channels, feature_dim)
        self.spatial_head = SpatialHead(feature_dim, max_strikes)
        self.temporal_head = TemporalHead(feature_dim, max_strikes)
        
        # Stop prediction head
        self.stop_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, max_strikes),  # Stop logit per slot
        )
    
    def forward(self, x, targets=None):
        """
        Forward pass.
        
        Args:
            x: (B, 4, 36, 192, 192) input images
            targets: unused (kept for consistent API)
        Returns:
            spatial_preds: (B, N, 2) - x, y coordinates
            temporal_preds: (B, N, 1) - t values
            stop_preds: (B, N) - stop logits
        """
        # Encode input
        features = self.encoder(x)                     # (B, feature_dim)
        
        # Predict coordinates
        spatial_preds = self.spatial_head(features)    # (B, N, 2) - x, y
        temporal_preds = self.temporal_head(features)  # (B, N, 1) - t
        
        # Predict stop logits
        stop_preds = self.stop_head(features)          # (B, N)
        
        return spatial_preds, temporal_preds, stop_preds
    
    @torch.no_grad()
    def generate(self, x, max_strikes=None, stop_threshold=0.5):
        """
        Generate variable-length predictions.
        
        Args:
            x: (B, 4, 36, 192, 192) input images
            max_strikes: unused (uses self.max_strikes)
            stop_threshold: probability threshold to stop
        Returns:
            List of (N_i, 3) tensors per sample with (t, x, y)
        """
        self.eval()
        spatial_preds, temporal_preds, stop_preds = self.forward(x)
        stop_probs = torch.sigmoid(stop_preds)
        
        # Combine predictions: (t, x, y)
        coords = torch.cat([temporal_preds, spatial_preds], dim=-1)  # (B, N, 3)
        
        results = []
        for i in range(coords.size(0)):
            # Find first stop position
            stops = stop_probs[i] > stop_threshold
            if stops.any():
                n_strikes = stops.float().argmax().item()
                if n_strikes == 0:
                    n_strikes = 1  # At least predict 1 strike
            else:
                n_strikes = self.max_strikes
            
            # Return normalized coordinates (denormalization done during evaluation)
            sample_coords = coords[i, :n_strikes]
            results.append(sample_coords.cpu())
        
        return results


# Parameter count
_model = DualCNNLightningPredictor()
print(f"DualCNNLightningPredictor parameters: {sum(p.numel() for p in _model.parameters()):,}")
del _model

## Loss Functions

Two loss functions are provided:

1. **Simple Masked Loss**: Same as original, easy to train, requires ordering alignment
2. **Chamfer Distance Loss**: Order-invariant, finds optimal assignment, may be harder to optimize

In [9]:
def compute_dual_decoder_loss(spatial_preds, temporal_preds, stop_preds, 
                               targets, mask, stop_weight=1.0):
    """
    Simple masked loss for Dual-CNN model.
    
    Args:
        spatial_preds: (B, N, 2) predicted (x, y)
        temporal_preds: (B, N, 1) predicted t
        stop_preds: (B, N) stop logits
        targets: (B, N, 3) target (t, x, y)
        mask: (B, N) True for real strikes, False for padding
        stop_weight: weight for stop loss
    Returns:
        total_loss, coord_loss, stop_loss
    """
    B, N, _ = targets.shape
    device = targets.device
    
    # Combine predictions: (t, x, y)
    strike_preds = torch.cat([temporal_preds, spatial_preds], dim=-1)  # (B, N, 3)
    
    # Regression loss on real strikes only (Smooth L1)
    coord_loss = F.smooth_l1_loss(strike_preds, targets, reduction='none')  # (B, N, 3)
    coord_loss = coord_loss.sum(dim=-1)  # (B, N)
    coord_loss = (coord_loss * mask.float()).sum() / (mask.sum() + 1e-8)
    
    # Stop token loss
    # Target: 0 for real strikes, 1 for the first padding position
    stop_targets = torch.zeros_like(stop_preds)
    for i in range(B):
        real_count = mask[i].sum().int()
        if real_count < N:
            stop_targets[i, real_count] = 1.0  # Stop at first padding position
    
    # Only compute stop loss on positions up to and including the first padding
    stop_mask = torch.zeros_like(stop_preds, dtype=torch.bool)
    for i in range(B):
        real_count = mask[i].sum().int()
        stop_mask[i, :min(real_count + 1, N)] = True
    
    stop_loss = F.binary_cross_entropy_with_logits(stop_preds, stop_targets, reduction='none')
    stop_loss = (stop_loss * stop_mask.float()).sum() / (stop_mask.sum() + 1e-8)
    
    total_loss = coord_loss + stop_weight * stop_loss
    
    return total_loss, coord_loss.item(), stop_loss.item()

In [10]:
def chamfer_distance_loss(pred_coords, gt_coords, pred_mask, gt_mask, 
                          lambda_spatial=1.0, lambda_time=2.0):
    """
    Chamfer distance loss for set-based supervision.
    Does not require ordering alignment between predictions and ground truth.
    
    Args:
        pred_coords: (B, N_pred, 3) - predicted (t, x, y) normalized to [0, 1]
        gt_coords: (B, N_gt, 3) - ground truth (t, x, y)
        pred_mask: (B, N_pred) - True for valid predictions
        gt_mask: (B, N_gt) - True for valid ground truth
        lambda_spatial: weight for spatial distance
        lambda_time: weight for temporal distance
    Returns:
        Chamfer distance loss (scalar)
    """
    B = pred_coords.shape[0]
    device = pred_coords.device
    total_loss = torch.tensor(0.0, device=device)
    valid_batches = 0
    
    for b in range(B):
        # Get valid predictions and targets
        pred_valid = pred_coords[b, pred_mask[b]]  # (M, 3)
        gt_valid = gt_coords[b, gt_mask[b]]        # (K, 3)
        
        M, K = len(pred_valid), len(gt_valid)
        
        if M == 0 or K == 0:
            # Penalize if predictions exist but no GT, or vice versa
            if M > 0 and K == 0:
                total_loss = total_loss + 1.0  # Penalty for false positives
                valid_batches += 1
            continue
        
        # Separate spatial and temporal components
        pred_time = pred_valid[:, 0:1]     # (M, 1)
        pred_spatial = pred_valid[:, 1:3]  # (M, 2)
        gt_time = gt_valid[:, 0:1]         # (K, 1)
        gt_spatial = gt_valid[:, 1:3]      # (K, 2)
        
        # Compute pairwise distances
        dist_spatial = torch.cdist(pred_spatial, gt_spatial, p=2)  # (M, K)
        dist_time = torch.cdist(pred_time, gt_time, p=1)           # (M, K)
        
        # Combined weighted distance
        dist = lambda_spatial * dist_spatial + lambda_time * dist_time  # (M, K)
        
        # Chamfer distance: pred→gt + gt→pred
        loss_p2g = dist.min(dim=1).values.mean()  # Each pred finds nearest gt
        loss_g2p = dist.min(dim=0).values.mean()  # Each gt finds nearest pred
        
        total_loss = total_loss + loss_p2g + loss_g2p
        valid_batches += 1
    
    if valid_batches == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)
    
    return total_loss / valid_batches


def compute_dual_decoder_chamfer_loss(spatial_preds, temporal_preds, stop_preds,
                                       targets, mask, stop_weight=1.0,
                                       lambda_spatial=1.0, lambda_time=2.0):
    """
    Combined loss using Chamfer distance for coordinates.
    
    Args:
        spatial_preds: (B, N, 2) predicted (x, y)
        temporal_preds: (B, N, 1) predicted t
        stop_preds: (B, N) stop logits
        targets: (B, N, 3) target (t, x, y)
        mask: (B, N) True for real strikes, False for padding
        stop_weight: weight for stop loss
        lambda_spatial: weight for spatial distance in Chamfer
        lambda_time: weight for temporal distance in Chamfer
    Returns:
        total_loss, coord_loss, stop_loss
    """
    B, N, _ = targets.shape
    device = targets.device
    
    # Combine predictions: (t, x, y)
    pred_coords = torch.cat([temporal_preds, spatial_preds], dim=-1)  # (B, N, 3)
    
    # For Chamfer, we need to determine which predictions are "active"
    # Use stop probability: predictions before the first stop are active
    stop_probs = torch.sigmoid(stop_preds)
    
    # Create prediction mask: True for positions before cumulative stop
    # Simple approach: use threshold to determine active predictions
    pred_mask = stop_probs < 0.5  # True if not stopping
    
    # For training stability, also include all positions up to max GT count
    for b in range(B):
        gt_count = mask[b].sum().int()
        pred_mask[b, :gt_count] = True  # Ensure we have enough active predictions
    
    # Chamfer loss on coordinates
    coord_loss = chamfer_distance_loss(
        pred_coords, targets, pred_mask, mask,
        lambda_spatial=lambda_spatial, lambda_time=lambda_time
    )
    
    # Stop token loss (same as simple loss)
    stop_targets = torch.zeros_like(stop_preds)
    for i in range(B):
        real_count = mask[i].sum().int()
        if real_count < N:
            stop_targets[i, real_count] = 1.0
    
    stop_mask = torch.zeros_like(stop_preds, dtype=torch.bool)
    for i in range(B):
        real_count = mask[i].sum().int()
        stop_mask[i, :min(real_count + 1, N)] = True
    
    stop_loss = F.binary_cross_entropy_with_logits(stop_preds, stop_targets, reduction='none')
    stop_loss = (stop_loss * stop_mask.float()).sum() / (stop_mask.sum() + 1e-8)
    
    total_loss = coord_loss + stop_weight * stop_loss
    
    return total_loss, coord_loss.item() if isinstance(coord_loss, torch.Tensor) else coord_loss, stop_loss.item()

## Dual-CNN Model Setup

In [ ]:
# Clear GPU memory from previous runs
import gc

# Delete any existing model/optimizer if they exist
for var in ['dual_model', 'model', 'optimizer', 'scheduler', '_model']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Memory before model: {torch.cuda.memory_allocated()/1024**2:.1f} MB allocated")

# Initialize Dual-CNN model
dual_model = DualCNNLightningPredictor(
    in_channels=4,
    feature_dim=512,
    max_strikes=50000  # Default: 50000, handles all events in dataset
).to(device)

print(f"Model parameters: {sum(p.numel() for p in dual_model.parameters()):,}")

if torch.cuda.is_available():
    print(f"GPU Memory after model: {torch.cuda.memory_allocated()/1024**2:.1f} MB allocated")

# Optimizer
optimizer = torch.optim.AdamW(dual_model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

## Training Loop

Set `USE_CHAMFER_LOSS = False` for simple masked loss (recommended to start),
or `USE_CHAMFER_LOSS = True` to use Chamfer distance loss.

In [ ]:
# ===== LOSS FUNCTION SELECTION =====
USE_CHAMFER_LOSS = False  # Set to True to use Chamfer distance loss
# ===================================

num_epochs = 20
train_losses = []
val_losses = []
train_coord_losses = []
train_stop_losses = []
best_val_loss = float('inf')

loss_name = "Chamfer" if USE_CHAMFER_LOSS else "Masked"
print(f"Training Dual-CNN with {loss_name} loss")

for epoch in range(num_epochs):
    # Training
    dual_model.train()
    total_train_loss = 0.0
    total_coord_loss = 0.0
    total_stop_loss = 0.0
    
    for batch_idx, (images, targets, mask, strike_counts, ids) in enumerate(train_loader):
        images = images.to(device)
        targets = targets.to(device)
        mask = mask.to(device)
        
        optimizer.zero_grad()
        
        spatial_preds, temporal_preds, stop_preds = dual_model(images, targets)
        
        if USE_CHAMFER_LOSS:
            loss, c_loss, s_loss = compute_dual_decoder_chamfer_loss(
                spatial_preds, temporal_preds, stop_preds, targets, mask
            )
        else:
            loss, c_loss, s_loss = compute_dual_decoder_loss(
                spatial_preds, temporal_preds, stop_preds, targets, mask
            )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dual_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        total_coord_loss += c_loss
        total_stop_loss += s_loss
    
    n_batches = len(train_loader)
    avg_train_loss = total_train_loss / n_batches
    train_losses.append(avg_train_loss)
    train_coord_losses.append(total_coord_loss / n_batches)
    train_stop_losses.append(total_stop_loss / n_batches)
    
    # Validation
    dual_model.eval()
    total_val_loss = 0.0
    
    with torch.no_grad():
        for images, targets, mask, strike_counts, ids in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            mask = mask.to(device)
            
            spatial_preds, temporal_preds, stop_preds = dual_model(images, targets)
            
            if USE_CHAMFER_LOSS:
                loss, _, _ = compute_dual_decoder_chamfer_loss(
                    spatial_preds, temporal_preds, stop_preds, targets, mask
                )
            else:
                loss, _, _ = compute_dual_decoder_loss(
                    spatial_preds, temporal_preds, stop_preds, targets, mask
                )
            total_val_loss += loss.item()
    
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    scheduler.step()
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(dual_model.state_dict(), 'best_dual_cnn_predictor.pth')
    
    # Plot progress
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    epochs_range = range(1, epoch + 2)
    
    axes[0].plot(epochs_range, train_losses, marker='o', label='Train')
    axes[0].plot(epochs_range, val_losses, marker='s', label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Total Loss')
    axes[0].set_title(f'Dual-CNN Total Loss ({loss_name})')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(epochs_range, train_coord_losses, marker='o', label='Coord Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Coordinate Loss')
    axes[1].set_title('Coordinate Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    axes[2].plot(epochs_range, train_stop_losses, marker='o', label='Stop Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Stop Loss')
    axes[2].set_title('Stop Token Loss')
    axes[2].legend()
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] (Dual-CNN with {loss_name} Loss)")
    print(f"  Train: {avg_train_loss:.4f} (Coord: {train_coord_losses[-1]:.4f}, Stop: {train_stop_losses[-1]:.4f})")
    print(f"  Val:   {avg_val_loss:.4f}")
    print(f"  LR:    {scheduler.get_last_lr()[0]:.6f}")

print("\nTraining complete!")

## Evaluation

In [ ]:
# Load best model
dual_model.load_state_dict(torch.load('best_dual_cnn_predictor.pth'))
dual_model.eval()

# Denormalization constants
TIME_SCALE = 10800.0  # seconds
SPATIAL_SCALE = 192.0  # pixels

def evaluate_dual_model(model, loader, loader_name):
    """Evaluate dual-CNN model on a data loader."""
    all_time_errors = []
    all_spatial_errors = []
    all_count_errors = []
    
    with torch.no_grad():
        for images, targets, mask, strike_counts, ids in loader:
            images = images.to(device)
            
            # Generate predictions
            predictions = model.generate(images, max_strikes=50000, stop_threshold=0.5)
            
            for i in range(len(ids)):
                actual_count = strike_counts[i].item()
                pred_strikes = predictions[i].numpy()  # (N_pred, 3)
                actual_strikes = targets[i, :actual_count].numpy()  # (N_actual, 3)
                
                pred_count = len(pred_strikes)
                all_count_errors.append(abs(pred_count - actual_count))
                
                if pred_count > 0 and actual_count > 0:
                    # Compare first min(pred, actual) strikes
                    n_compare = min(pred_count, actual_count)
                    
                    # Denormalize
                    pred_t = pred_strikes[:n_compare, 0] * TIME_SCALE
                    pred_x = pred_strikes[:n_compare, 1] * SPATIAL_SCALE
                    pred_y = pred_strikes[:n_compare, 2] * SPATIAL_SCALE
                    
                    actual_t = actual_strikes[:n_compare, 0] * TIME_SCALE
                    actual_x = actual_strikes[:n_compare, 1] * SPATIAL_SCALE
                    actual_y = actual_strikes[:n_compare, 2] * SPATIAL_SCALE
                    
                    # Errors
                    time_error = np.abs(pred_t - actual_t)
                    spatial_error = np.sqrt((pred_x - actual_x)**2 + (pred_y - actual_y)**2)
                    
                    all_time_errors.extend(time_error)
                    all_spatial_errors.extend(spatial_error)
    
    print(f"\n{loader_name} Results (Dual-CNN):")
    print(f"  Mean Time Error: {np.mean(all_time_errors):.1f} seconds ({np.mean(all_time_errors)/60:.1f} minutes)")
    print(f"  Mean Spatial Error: {np.mean(all_spatial_errors):.1f} pixels")
    print(f"  Mean Count Error: {np.mean(all_count_errors):.1f} strikes")
    
    return all_time_errors, all_spatial_errors, all_count_errors

# Evaluate
val_results = evaluate_dual_model(dual_model, val_loader, "Validation")
test_results = evaluate_dual_model(dual_model, test_loader, "Test")

## Visualization: Predictions vs Actual

In [ ]:
# Visualize predictions vs actual for test samples
dual_model.eval()

with torch.no_grad():
    for images, targets, mask, strike_counts, ids in test_loader:
        images_device = images.to(device)
        predictions = dual_model.generate(images_device, max_strikes=50000, stop_threshold=0.5)
        
        for i in range(min(4, len(ids))):
            actual_count = strike_counts[i].item()
            pred_strikes = predictions[i].numpy()
            actual_strikes = targets[i, :actual_count].numpy()
            
            # Denormalize
            pred_denorm = pred_strikes.copy()
            pred_denorm[:, 0] *= TIME_SCALE
            pred_denorm[:, 1] *= SPATIAL_SCALE
            pred_denorm[:, 2] *= SPATIAL_SCALE
            
            actual_denorm = actual_strikes.copy()
            actual_denorm[:, 0] *= TIME_SCALE
            actual_denorm[:, 1] *= SPATIAL_SCALE
            actual_denorm[:, 2] *= SPATIAL_SCALE
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle(f'Event: {ids[i]} | Actual: {actual_count} strikes, Predicted: {len(pred_strikes)} strikes', 
                        fontsize=12)
            
            # Spatial plot
            axes[0].scatter(actual_denorm[:, 1], actual_denorm[:, 2], 
                           alpha=0.6, label=f'Actual ({actual_count})', c='blue', s=15)
            if len(pred_denorm) > 0:
                axes[0].scatter(pred_denorm[:, 1], pred_denorm[:, 2], 
                               alpha=0.6, label=f'Predicted ({len(pred_strikes)})', c='red', s=15, marker='x')
            axes[0].set_xlabel('X coordinate (pixels)')
            axes[0].set_ylabel('Y coordinate (pixels)')
            axes[0].set_title('Spatial Distribution')
            axes[0].legend()
            axes[0].set_xlim(0, 192)
            axes[0].set_ylim(192, 0)
            axes[0].grid(True, alpha=0.3)
            
            # Temporal histogram
            bins = np.arange(0, 36 * 5 + 1, 5)  # 5-minute bins
            if actual_count > 0:
                axes[1].hist(actual_denorm[:, 0] / 60.0, bins=bins, alpha=0.6, 
                            label='Actual', color='blue', edgecolor='black')
            if len(pred_denorm) > 0:
                axes[1].hist(pred_denorm[:, 0] / 60.0, bins=bins, alpha=0.6,
                            label='Predicted', color='red', edgecolor='black')
            axes[1].set_xlabel('Time (minutes)')
            axes[1].set_ylabel('Number of Strikes')
            axes[1].set_title('Temporal Distribution (5-min bins)')
            axes[1].legend()
            axes[1].set_xlim(0, 180)
            axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
        break

## Test Model Outputs

Quick test to verify model forward pass and output shapes, with visualization of spatial (x, y) and temporal distributions.

In [ ]:
# Test model outputs with a single batch
dual_model.eval()

# Denormalization constants
TIME_SCALE = 10800.0  # seconds
SPATIAL_SCALE = 192.0  # pixels

with torch.no_grad():
    # Get a single batch from test loader
    images, targets, mask, strike_counts, ids = next(iter(test_loader))
    images_device = images.to(device)
    
    # Forward pass
    spatial_preds, temporal_preds, stop_preds = dual_model(images_device)
    
    print("Model Output Shapes:")
    print(f"  Spatial preds:  {spatial_preds.shape}  (expected: [B, max_strikes, 2])")
    print(f"  Temporal preds: {temporal_preds.shape}  (expected: [B, max_strikes, 1])")
    print(f"  Stop preds:     {stop_preds.shape}  (expected: [B, max_strikes])")
    
    # Test generate method
    predictions = dual_model.generate(images_device, stop_threshold=0.5)
    print(f"\nGenerated predictions: {len(predictions)} samples")
    for i, pred in enumerate(predictions):
        print(f"  Sample {i}: {pred.shape[0]} strikes predicted (actual: {strike_counts[i].item()})")
    
    # Visualize first sample
    idx = 0
    pred_strikes = predictions[idx].numpy()
    actual_count = strike_counts[idx].item()
    actual_strikes = targets[idx, :actual_count].numpy()
    
    # Denormalize
    pred_denorm = pred_strikes.copy()
    pred_denorm[:, 0] *= TIME_SCALE
    pred_denorm[:, 1] *= SPATIAL_SCALE
    pred_denorm[:, 2] *= SPATIAL_SCALE
    
    actual_denorm = actual_strikes.copy()
    actual_denorm[:, 0] *= TIME_SCALE
    actual_denorm[:, 1] *= SPATIAL_SCALE
    actual_denorm[:, 2] *= SPATIAL_SCALE
    
    # Create two plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Event: {ids[idx]} | Actual: {actual_count} strikes, Predicted: {len(pred_strikes)} strikes', 
                fontsize=12)
    
    # Plot 1: Spatial distribution (x, y)
    axes[0].scatter(actual_denorm[:, 1], actual_denorm[:, 2], 
                   alpha=0.6, label=f'Actual ({actual_count})', c='blue', s=15)
    if len(pred_denorm) > 0:
        axes[0].scatter(pred_denorm[:, 1], pred_denorm[:, 2], 
                       alpha=0.6, label=f'Predicted ({len(pred_strikes)})', c='red', s=15, marker='x')
    axes[0].set_xlabel('X coordinate (pixels)')
    axes[0].set_ylabel('Y coordinate (pixels)')
    axes[0].set_title('Spatial Distribution (x, y)')
    axes[0].legend()
    axes[0].set_xlim(0, 192)
    axes[0].set_ylim(192, 0)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Temporal sequence (all events)
    if actual_count > 0:
        axes[1].scatter(actual_denorm[:, 0] / 60.0, np.arange(len(actual_denorm)), 
                       alpha=0.6, label='Actual', c='blue', s=15)
    if len(pred_denorm) > 0:
        axes[1].scatter(pred_denorm[:, 0] / 60.0, np.arange(len(pred_denorm)),
                       alpha=0.6, label='Predicted', c='red', s=15, marker='x')
    axes[1].set_xlabel('Time (minutes)')
    axes[1].set_ylabel('Event Index')
    axes[1].set_title('Temporal Sequence (all events)')
    axes[1].legend()
    axes[1].set_xlim(0, 180)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()